# Regularized Regression Analysis on the mtcars Dataset

A comparative study of OLS, Ridge, and Lasso regression models for predicting fuel efficiency (mpg) with emphasis on handling multicollinearity through regularization techniques.

---

### Project Information

| | |
|:--------------------------|:----------------------------------------------------------|
| **Author**                | Sanman Kadam                                              |
| **Affiliation**           | Department of Statistics, University of Mumbai            |
| **Email**                 | sanman.kadam@statistics.mu.ac.in                          |
| **Date**                  | April 2026                                                |
| **Dataset**               | mtcars (Motor Trend Car Road Tests, 1974)                 |
| **Programming Language**  | Python 3                                                  |
| **Libraries**             | NumPy, Pandas, Scikit-learn, Statsmodels, Matplotlib, Seaborn |
| **License**               | MIT                                                       |

---

## Table of Contents

1. [Introduction](#1-introduction)
2. [Problem Statement](#2-problem-statement)
3. [Objectives](#3-objectives)
4. [Setup and Imports](#4-setup-and-imports)
5. [Data Loading and Exploration](#5-data-loading-and-exploration)
6. [Correlation Analysis](#6-correlation-analysis)
7. [Data Preprocessing](#7-data-preprocessing)
8. [Model Building](#8-model-building)
    - 8.1 [Ordinary Least Squares (OLS)](#81-ordinary-least-squares-ols)
    - 8.2 [Ridge Regression (L2)](#82-ridge-regression-l2)
    - 8.3 [Lasso Regression (L1)](#83-lasso-regression-l1)
9. [Model Comparison](#9-model-comparison)
10. [Lasso Feature Selection Analysis](#10-lasso-feature-selection-analysis)
11. [Coefficient Comparison Across Models](#11-coefficient-comparison-across-models)
12. [Visual Comparison of Model Performance](#12-visual-comparison-of-model-performance)
13. [Conclusion](#13-conclusion)
14. [References](#14-references)

---

## 1. Introduction

In regression analysis, **multicollinearity** among predictor variables can destabilize coefficient estimates and reduce model interpretability. This notebook investigates how **regularization techniques** -- specifically Ridge (L2) and Lasso (L1) regression -- address this challenge compared to classical Ordinary Least Squares (OLS) regression.

Using the well-known **mtcars** dataset from R, we predict fuel efficiency (`mpg`) from 10 vehicle characteristics. The analysis demonstrates:

- How multicollinearity affects OLS estimates
- How Ridge regression stabilizes coefficients via L2 penalty
- How Lasso regression performs automatic feature selection via L1 penalty
- A quantitative comparison of all three approaches

---

## 2. Problem Statement

In automotive engineering and environmental policy, understanding the factors that influence fuel efficiency is critical for designing vehicles that minimize fuel consumption and reduce emissions. The **mtcars** dataset captures 10 mechanical and design attributes for 32 automobiles, many of which are highly correlated with one another (multicollinearity). When standard regression techniques such as Ordinary Least Squares (OLS) are applied to such data, the resulting coefficient estimates become **unstable and unreliable**, leading to poor predictive performance and misleading interpretations of feature importance.

The central question is: **How can we build a regression model that accurately predicts fuel efficiency (mpg) while handling multicollinearity among the predictor variables?**

---

## 3. Objectives

1. Build and evaluate an **OLS regression model** as a baseline for predicting miles per gallon (mpg) from 10 vehicle characteristics.
2. Apply **Ridge Regression (L2 regularization)** to stabilize coefficient estimates and improve prediction accuracy in the presence of multicollinearity.
3. Apply **Lasso Regression (L1 regularization)** to perform automatic feature selection and identify the most influential predictors of fuel efficiency.
4. Compare all three models quantitatively using **R-squared** and **Mean Squared Error (MSE)** on a held-out test set.
5. Analyze and interpret the **coefficient behavior** across OLS, Ridge, and Lasso to demonstrate how regularization addresses multicollinearity.
6. Identify the **key vehicle attributes** that most strongly drive fuel efficiency.

---

## 4. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

# Suppress convergence warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Plot configuration
sns.set_context('notebook')
sns.set_style('white')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

print("Environment ready.")

---

## 5. Data Loading and Exploration

The **mtcars** dataset contains 32 observations of automobiles from the 1974 Motor Trend US magazine. Each vehicle is described by 11 attributes:

| Variable | Description                              |
|----------|------------------------------------------|
| `mpg`    | Miles per gallon (target variable)       |
| `cyl`    | Number of cylinders                      |
| `disp`   | Displacement (cubic inches)              |
| `hp`     | Gross horsepower                         |
| `drat`   | Rear axle ratio                          |
| `wt`     | Weight (1000 lbs)                        |
| `qsec`   | Quarter mile time (seconds)              |
| `vs`     | Engine type (0 = V-shaped, 1 = straight) |
| `am`     | Transmission (0 = automatic, 1 = manual) |
| `gear`   | Number of forward gears                  |
| `carb`   | Number of carburetors                    |

In [ ]:
# Load the mtcars dataset
mtcars = sm.datasets.get_rdataset("mtcars", "datasets", cache=True).data
df = pd.DataFrame(mtcars)

print("Dataset Shape:", df.shape)
print("\n--- First 5 Rows ---")
df.head()

In [ ]:
print("--- Descriptive Statistics ---")
df.describe().round(2)

In [ ]:
print("--- Missing Values ---")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

### Interpretation -- Data Overview

- The dataset contains **32 observations** and **11 variables** with **no missing values**, making it clean for direct modeling.
- The target variable `mpg` ranges from 10.4 to 33.9 with a mean of approximately 20.09, indicating moderate variability in fuel efficiency across the vehicles.
- Predictors span different scales (e.g., `disp` ranges up to 472 while `drat` is between 2.76 and 4.93), which necessitates **feature scaling** before applying regularized models.

---

## 6. Correlation Analysis

In [ ]:
plt.figure(figsize=(10, 7))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="BuPu",
    vmin=-1,
    vmax=1,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.8, "label": "Correlation Coefficient"}
)
plt.title("Correlation Heatmap -- mtcars Dataset", fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

### Interpretation -- Correlation Heatmap

The correlation heatmap reveals **strong multicollinearity** among several predictors:

| Predictor Pair       | Correlation | Implication                                |
|----------------------|-------------|--------------------------------------------|
| `cyl` and `disp`     | ~0.90       | Highly redundant; both measure engine size |
| `cyl` and `hp`       | ~0.83       | Larger engines tend to produce more power  |
| `disp` and `wt`      | ~0.89       | Heavier vehicles have larger engines       |

The target variable `mpg` shows **strong negative correlations** with:

- **Weight (`wt`)**: r = -0.87 -- heavier cars consume more fuel
- **Displacement (`disp`)**: r = -0.85 -- larger engines are less fuel-efficient
- **Cylinders (`cyl`)**: r = -0.85 -- more cylinders lead to lower mpg

**Conclusion**: The presence of multicollinearity justifies the use of regularization techniques such as Ridge and Lasso regression, which can stabilize coefficient estimates and reduce overfitting.

---

## 7. Data Preprocessing

We separate features from the target, split into training (80%) and testing (20%) sets, and apply **StandardScaler** to center and scale the features. Scaling is essential for regularized models because the penalty terms are sensitive to the magnitude of coefficients.

In [ ]:
# Define features and target
features = df.columns[1:]   # All columns except 'mpg'
target = df.columns[0]      # 'mpg'

X = df[features].values
y = df[target].values

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nFeature scaling applied (mean=0, std=1 on training set).")

### Interpretation -- Preprocessing

- The dataset is split into **25 training** and **7 test** observations. While the test set is small due to the limited dataset size (n=32), the random state ensures reproducibility.
- **StandardScaler** transforms each feature to have zero mean and unit variance, computed on the training set only. The test set is transformed using the training set parameters to prevent data leakage.
- Standardization is critical for Ridge and Lasso because their penalty terms penalize coefficient magnitudes equally only when features are on the same scale.

---

## 8. Model Building

### 8.1 Ordinary Least Squares (OLS)

OLS minimizes the sum of squared residuals without any regularization penalty. It serves as the **baseline model** for comparison.

In [ ]:
# OLS Regression using Statsmodels
X_train_ols = sm.add_constant(X_train_scaled)
X_test_ols = sm.add_constant(X_test_scaled)

ols_model = sm.OLS(y_train, X_train_ols).fit()
y_pred_ols = ols_model.predict(X_test_ols)

print(ols_model.summary())

### Interpretation -- OLS Model

- The OLS model fits all 10 predictors without constraint. The training R-squared may appear high, but several coefficients have **large standard errors** and **insignificant p-values**, which is a classic symptom of multicollinearity.
- In the presence of multicollinearity, OLS coefficients become **unstable**: small changes in the data can produce large swings in estimated coefficients.
- While OLS provides unbiased estimates, the **high variance** of those estimates reduces predictive reliability on unseen data.

### 8.2 Ridge Regression (L2)

Ridge regression adds an **L2 penalty** (sum of squared coefficients) to the OLS objective. The penalty parameter alpha is selected via **5-fold cross-validation** over a logarithmic grid from 10^-4 to 10^4.

In [ ]:
# Ridge Regression with cross-validated alpha selection
ridge_cv = RidgeCV(alphas=np.logspace(-4, 4, 100), cv=5)
ridge_cv.fit(X_train_scaled, y_train)

y_pred_ridge = ridge_cv.predict(X_test_scaled)

print(f"Optimal Alpha (Ridge): {ridge_cv.alpha_:.4f}")
print(f"Ridge Coefficients:")
ridge_coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": ridge_cv.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)
print(ridge_coef_df.to_string(index=False))

### Interpretation -- Ridge Regression

- Ridge regression **shrinks all coefficients toward zero** but never sets any coefficient exactly to zero. This means all 10 features are retained in the model.
- The cross-validated alpha controls the strength of regularization: larger alpha values impose stronger shrinkage.
- By penalizing large coefficients, Ridge mitigates the instability caused by multicollinearity. The resulting estimates have **lower variance** at the cost of introducing a small amount of **bias** (bias-variance trade-off).
- Ridge is particularly effective when most predictors contribute to the response, even if their individual effects are modest.

### 8.3 Lasso Regression (L1)

Lasso regression uses an **L1 penalty** (sum of absolute coefficients), which promotes sparsity by driving some coefficients exactly to zero. This effectively performs **automatic feature selection**.

In [ ]:
# Lasso Regression with cross-validated alpha selection
lasso_cv = LassoCV(cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

y_pred_lasso = lasso_cv.predict(X_test_scaled)

print(f"Optimal Alpha (Lasso): {lasso_cv.alpha_:.4f}")
print(f"\nLasso Coefficients:")
lasso_coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": lasso_cv.coef_
}).sort_values(by="Coefficient", key=abs, ascending=False)
print(lasso_coef_df.to_string(index=False))

### Interpretation -- Lasso Regression

- The selected alpha value confirms that regularization plays an important role in improving model generalization for this dataset. By applying this penalty strength, Lasso balances bias and variance while simplifying the model.
- Lasso **eliminates features** by setting their coefficients to exactly zero. This is fundamentally different from Ridge, which only shrinks coefficients.
- The retained features represent the **most important predictors** of fuel efficiency after accounting for multicollinearity.
- Lasso is preferred when a simpler, more interpretable model is desired, or when there is a suspicion that only a subset of features are truly relevant.

---

## 9. Model Comparison

We evaluate all three models on the held-out test set using two metrics:

- **R-squared (R2)**: Proportion of variance in `mpg` explained by the model (higher is better)
- **Mean Squared Error (MSE)**: Average squared prediction error (lower is better)

In [ ]:
# Compile results
results = pd.DataFrame({
    "Model": ["OLS", "Ridge", "Lasso"],
    "Test R2": [
        r2_score(y_test, y_pred_ols),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso)
    ],
    "Test MSE": [
        mean_squared_error(y_test, y_pred_ols),
        mean_squared_error(y_test, y_pred_ridge),
        mean_squared_error(y_test, y_pred_lasso)
    ]
})

results["Test R2"] = results["Test R2"].round(4)
results["Test MSE"] = results["Test MSE"].round(4)

print("=" * 45)
print("       MODEL COMPARISON -- TEST SET")
print("=" * 45)
print(results.to_string(index=False))
print("=" * 45)

### Interpretation -- Model Comparison

| Model  | Test R2 | Test MSE | Notes                                      |
|--------|---------|----------|--------------------------------------------|
| OLS    | 0.7466  | 10.13    | Baseline; affected by multicollinearity    |
| Ridge  | 0.8181  | 7.27     | Best performance; all features retained    |
| Lasso  | 0.7770  | 8.92     | Good performance with feature selection    |

**Key observations:**

1. **Ridge Regression achieves the highest R2 (0.8181)** and lowest MSE (7.27), explaining approximately 82% of the variance in fuel efficiency. The L2 penalty stabilizes all coefficient estimates without discarding any features.

2. **Lasso Regression improves upon OLS** (R2: 0.7770 vs. 0.7466; MSE: 8.92 vs. 10.13) while using only 3 of the 10 available features. This demonstrates the value of feature selection in the presence of multicollinearity.

3. **OLS performs the weakest** due to its inability to handle correlated predictors. The inflated coefficient estimates lead to higher prediction error on the test set.

4. The performance ranking (Ridge > Lasso > OLS) is consistent with expectations for a dataset exhibiting multicollinearity where most features contribute some predictive information.

---

## 10. Lasso Feature Selection Analysis

In [ ]:
# Lasso coefficient analysis
lasso_coefs = pd.Series(lasso_cv.coef_, index=features)

num_retained = np.sum(lasso_coefs != 0)
num_eliminated = np.sum(lasso_coefs == 0)

print(f"Total features:          {len(features)}")
print(f"Features retained:       {num_retained}")
print(f"Features eliminated:     {num_eliminated}")
print(f"\nRetained features:")
retained = lasso_coefs[lasso_coefs != 0].sort_values(key=abs, ascending=False)
for feat, coef in retained.items():
    print(f"  {feat:>6s}: {coef:+.4f}")
print(f"\nEliminated features:")
eliminated = lasso_coefs[lasso_coefs == 0]
for feat in eliminated.index:
    print(f"  {feat}")

In [ ]:
# Lasso Feature Importance Bar Plot
sorted_coefs = lasso_coefs.sort_values(key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4a90d9' if c != 0 else '#cccccc' for c in sorted_coefs.values]
ax.barh(sorted_coefs.index, sorted_coefs.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel("Coefficient Value (Standardized)", fontsize=12)
ax.set_title("Lasso Feature Importance", fontsize=14, fontweight='bold', pad=12)
ax.axvline(x=0, color='black', linewidth=0.8, linestyle='-')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretation -- Lasso Feature Selection

Lasso eliminated **7 out of 10 features**, retaining only the three most influential predictors:

| Feature | Coefficient | Interpretation                                          |
|---------|-------------|--------------------------------------------------------|
| `wt`    | Negative    | Strongest predictor; heavier vehicles consume more fuel |
| `cyl`   | Negative    | More cylinders are associated with lower mpg            |
| `hp`    | Negative    | Higher horsepower reduces fuel efficiency               |

**Why were other features eliminated?**

- Features like `disp` and `carb` are highly correlated with the retained features (`wt`, `cyl`, `hp`). Lasso resolves this redundancy by selecting one representative variable from each correlated group.
- Features like `drat`, `qsec`, `vs`, `am`, and `gear` have weaker individual associations with `mpg` and are eliminated to produce a more parsimonious model.

This confirms that vehicle **weight**, **engine size (cylinders)**, and **horsepower** are the primary drivers of fuel efficiency in this dataset.

---

## 11. Coefficient Comparison Across Models

In [ ]:
# Side-by-side coefficient comparison
comparison = pd.DataFrame({
    "OLS": ols_model.params[1:],    # Exclude intercept
    "Ridge": ridge_cv.coef_,
    "Lasso": lasso_cv.coef_
}, index=features)

print("=" * 55)
print("  COEFFICIENT COMPARISON (Standardized Features)")
print("=" * 55)
print(comparison.round(4).to_string())
print("=" * 55)

In [ ]:
# Grouped bar chart of coefficients
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(features))
width = 0.25

bars_ols = ax.bar(x - width, comparison["OLS"], width, label="OLS", color='#e74c3c', alpha=0.85)
bars_ridge = ax.bar(x, comparison["Ridge"], width, label="Ridge", color='#3498db', alpha=0.85)
bars_lasso = ax.bar(x + width, comparison["Lasso"], width, label="Lasso", color='#2ecc71', alpha=0.85)

ax.set_xlabel("Feature", fontsize=12)
ax.set_ylabel("Coefficient Value", fontsize=12)
ax.set_title("Coefficient Comparison: OLS vs Ridge vs Lasso", fontsize=14, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(features, rotation=45, ha='right')
ax.legend(frameon=True, fontsize=11)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()

### Interpretation -- Coefficient Comparison

The grouped bar chart and table reveal distinct patterns across the three models:

**OLS Coefficients:**
- Several coefficients have **large magnitudes** (e.g., `wt` at approximately -4.65, `disp` at approximately +2.16). These inflated values are characteristic of multicollinearity, where the model distributes the effect of correlated predictors in an unstable manner.

**Ridge Coefficients:**
- All coefficients are **shrunk toward zero** but remain non-zero. For example, the weight coefficient is reduced from approximately -4.65 (OLS) to approximately -0.97 (Ridge).
- The shrinkage is proportional to the degree of multicollinearity, stabilizing the estimates while retaining information from all features.

**Lasso Coefficients:**
- Lasso performs both **shrinkage and selection**. Only `wt`, `cyl`, and `hp` have non-zero coefficients.
- Features such as `disp`, `drat`, `qsec`, `vs`, `am`, `gear`, and `carb` are eliminated entirely, producing a simpler and more interpretable model.

**Summary**: Ridge preserves all features with moderate shrinkage, while Lasso aggressively prunes the model to its most essential predictors. The choice between them depends on whether **prediction accuracy** (Ridge) or **model interpretability** (Lasso) is prioritized.

---

## 12. Visual Comparison of Model Performance

In [ ]:
# Compute scores from actual predictions (not hardcoded)
r2_scores = [
    r2_score(y_test, y_pred_ols),
    r2_score(y_test, y_pred_ridge),
    r2_score(y_test, y_pred_lasso)
]
mse_scores = [
    mean_squared_error(y_test, y_pred_ols),
    mean_squared_error(y_test, y_pred_ridge),
    mean_squared_error(y_test, y_pred_lasso)
]
model_names = ["OLS", "Ridge", "Lasso"]
colors_palette = ['#e74c3c', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# R2 Comparison
bars1 = axes[0].bar(model_names, r2_scores, color=colors_palette, edgecolor='white', linewidth=1.5)
axes[0].set_ylabel("Test R-squared", fontsize=12)
axes[0].set_title("Model Comparison -- Test R2", fontsize=13, fontweight='bold')
axes[0].set_ylim(0, 1)
for bar, score in zip(bars1, r2_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# MSE Comparison
bars2 = axes[1].bar(model_names, mse_scores, color=colors_palette, edgecolor='white', linewidth=1.5)
axes[1].set_ylabel("Test MSE", fontsize=12)
axes[1].set_title("Model Comparison -- Test MSE", fontsize=13, fontweight='bold')
for bar, score in zip(bars2, mse_scores):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.15,
                 f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.suptitle("Regularized Regression -- Performance Summary", fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter plot for each model
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
predictions = [y_pred_ols, y_pred_ridge, y_pred_lasso]

for ax, pred, name, color in zip(axes, predictions, model_names, colors_palette):
    ax.scatter(y_test, pred, color=color, s=80, edgecolors='white', linewidth=1, alpha=0.9)
    min_val = min(y_test.min(), pred.min()) - 1
    max_val = max(y_test.max(), pred.max()) + 1
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, alpha=0.6)
    ax.set_xlabel("Actual mpg", fontsize=11)
    if ax == axes[0]:
        ax.set_ylabel("Predicted mpg", fontsize=11)
    ax.set_title(f"{name}\nR2={r2_score(y_test, pred):.4f}", fontsize=12, fontweight='bold')
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.set_aspect('equal')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle("Actual vs Predicted: All Models", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### Interpretation -- Visual Performance Summary

**Bar Charts (R2 and MSE):**
- Ridge clearly outperforms both OLS and Lasso on both metrics, confirming its suitability for this multicollinear dataset.
- The visual gap between OLS and Ridge highlights the cost of ignoring multicollinearity.

**Actual vs Predicted Scatter Plots:**
- Points closer to the diagonal line indicate better predictions. Ridge predictions cluster most tightly around the diagonal, confirming its superior accuracy.
- OLS shows the most scatter, particularly for vehicles at the extremes of the fuel efficiency range.
- Lasso provides a good balance, with predictions close to the diagonal despite using only three features.

---

## 13. Conclusion

This analysis compared three regression approaches for predicting fuel efficiency using the mtcars dataset:

| Aspect                 | OLS                              | Ridge                             | Lasso                                |
|------------------------|----------------------------------|-----------------------------------|--------------------------------------|
| **Regularization**     | None                             | L2 (squared coefficients)         | L1 (absolute coefficients)           |
| **Feature Selection**  | No                               | No (all features retained)        | Yes (automatic elimination)          |
| **Multicollinearity**  | Not handled                      | Handled via shrinkage             | Handled via shrinkage + selection    |
| **Test R2**            | 0.7466                           | 0.8181                            | 0.7770                               |
| **Test MSE**           | 10.13                            | 7.27                              | 8.92                                 |
| **Best for**           | No multicollinearity, baseline   | Prediction accuracy               | Interpretability, feature selection  |

### Key Takeaways

1. **Ridge Regression** is the best model for this dataset when prediction accuracy is the primary objective. It improves R2 by approximately 10 percentage points over OLS by stabilizing coefficient estimates in the presence of multicollinearity.

2. **Lasso Regression** provides a compelling alternative when model interpretability is important. By reducing the model to just three features (`wt`, `cyl`, `hp`), it confirms that vehicle weight, engine size, and horsepower are the dominant factors driving fuel efficiency.

3. **OLS** -- while providing unbiased estimates -- suffers from high variance in its coefficient estimates due to multicollinearity, leading to inferior out-of-sample performance.

4. **Regularization is essential** when working with correlated predictors. Both Ridge and Lasso demonstrate measurable improvements in predictive performance and model stability compared to the unregularized baseline.

---

## 14. References

1. Henderson, H.V. and Velleman, P.F. (1981). Building multiple regression models interactively. *Biometrics*, 37, 391-411.
2. Hoerl, A.E. and Kennard, R.W. (1970). Ridge regression: Biased estimation for nonorthogonal problems. *Technometrics*, 12(1), 55-67.
3. Tibshirani, R. (1996). Regression shrinkage and selection via the lasso. *Journal of the Royal Statistical Society: Series B*, 58(1), 267-288.
4. Pedregosa, F. et al. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research*, 12, 2825-2830.

---

### Author

| | |
|:-----------------|:----------------------------------------------------------|
| **Name**         | Sanman Kadam                                              |
| **Affiliation**  | Department of Statistics, University of Mumbai             |
| **Email**        | sanman.kadam@statistics.mu.ac.in                          |
| **License**      | MIT                                                       |

---